# Lab 01: How a Spark application runs (reference solution)

**Time:** about 2 h 30 min.

**Checklist questions this lab gives you evidence for:**<br><span style="color:#d93025"><b>1.1 Apache Spark architecture</b></span><br><span style="color:#d93025"><b>1.2 What is a SparkSession?</b></span><br><span style="color:#d93025"><b>1.3 Driver vs. executor</b></span><br><span style="color:#d93025"><b>1.5 Narrow vs. wide dependencies</b></span><br><span style="color:#d93025"><b>1.7 Fault tolerance</b></span><br><span style="color:#d93025"><b>1.10 Lifecycle of a Spark job</b></span><br><span style="color:#d93025"><b>1.11 Transformation vs. action</b></span><br><span style="color:#d93025"><b>1.12 Lazy evaluation</b></span>

**Goal:** see with your own eyes when Spark runs a job, how a query becomes stages and tasks, and what happens when a task fails. The last section is DataFrame coding practice.

## Data information
A small online shop. The setup cell below creates three tables in `data/` (seeded, so everyone gets the same rows):

**orders** (`data/orders.parquet`, 1,000,000 rows, January to March 2024)
* *order_id*, *customer_id* (empty for guest checkout), *product_id*, *quantity* (1 to 5)
* *amount*: quantity × product price, in dollars
* *status*: paid, shipped, cancelled or returned; *payment_method*: card, cash or wallet
* *order_ts*, *order_date*: when the order was placed
* *tracking_code*: a 32-character code

**customers** (`data/customers.csv`, 10,000 rows): *customer_id*, *name*, *country*, *signup_date*, *segment* (retail or business). Customer 1 is "Big Reseller Ltd", who places 30% of all orders.

**products** (`data/products.csv`, 200 rows): *product_id*, *name*, *category*, *price*

## How to work
* Write your answer in the code cell under each question, then run the check cell below it: ✅ correct, ❌ try again.
* When a question says *guess first*, write your guess as a comment before you run anything.
* Keep the Spark UI open (the setup cell prints its address).
* Stuck for more than 10 minutes? Read the documentation list at the top, then the reference solution.


## Documentation
Everything you need is on these pages. The questions do not repeat them: read, try things out in a cell, and find the answers yourself.

* [SparkSession](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.SparkSession.html)
* [Cluster Mode Overview](https://spark.apache.org/docs/latest/cluster-overview.html)
* [DataFrameReader.parquet](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameReader.parquet.html)
* [DataFrameReader.csv](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameReader.csv.html)
* [DataFrame.withColumn](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.withColumn.html)
* [RDD operations](https://spark.apache.org/docs/latest/rdd-programming-guide.html#rdd-operations)
* [DataFrame.explain](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.explain.html)
* [DataFrame.agg](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.agg.html)
* [functions.avg](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.avg.html)
* [Spark UI: Stages tab](https://spark.apache.org/docs/latest/web-ui.html#stages-tab)
* [DataFrame.orderBy](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.orderBy.html)
* [DataFrame.groupBy](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.groupBy.html)
* [Tuning partitions](https://spark.apache.org/docs/latest/sql-performance-tuning.html#tuning-partitions)
* [RDD.getNumPartitions](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.getNumPartitions.html)
* [DataFrame.join](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html)
* [Submitting applications](https://spark.apache.org/docs/latest/submitting-applications.html)
* [Self-contained applications](https://spark.apache.org/docs/latest/quick-start.html#self-contained-applications)
* [functions.when](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.when.html)
* [functions.regexp_extract](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.regexp_extract.html)
* [functions.dayofweek](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.dayofweek.html)
* [ANSI mode](https://spark.apache.org/docs/latest/sql-ref-ansi-compliance.html)
* [DataFrame.na.fill](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameNaFunctions.fill.html)
* [Column.isNull](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Column.isNull.html)
* [functions.udf](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.udf.html)
* [Master URLs](https://spark.apache.org/docs/latest/submitting-applications.html#master-urls)
* [spark.task.maxFailures](https://spark.apache.org/docs/latest/configuration.html#scheduling)

## Setup
Run the cell below once (its code is hidden: click it to expand). It creates the data and starts Spark.

In [ ]:
# Setup: run this cell once. It creates the shop data in data/ and starts Spark.
# Nothing to solve here, and you do not need to read it.
import contextlib, hashlib, io, json, os, re, shutil, sys, time, urllib.request
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this kernel's Python
from pyspark.sql import SparkSession, Window, functions as F

# ---------------------------------------------------------------- the shop data ---------
# A small online shop, generated from seeded hash functions: every machine gets the same rows.
COUNTRIES = ["Uzbekistan", "Germany", "USA", "India", "Brazil", "Japan"]
CATEGORIES = ["books", "toys", "garden", "sports", "kitchen", "beauty", "music", "office"]
N_CUSTOMERS, N_PRODUCTS, N_ORDERS = 10_000, 200, 1_000_000

def _u(col, salt):
    """Pseudo-random number in [0, 1) computed from a column (same result on every machine)."""
    return F.pmod(F.xxhash64(col, F.lit(salt)), F.lit(1_000_000)) / 1_000_000.0

def _pick(u, options):
    return F.element_at(F.array(*[F.lit(o) for o in options]), (F.floor(u * len(options)) + 1).cast("int"))

def make_customers(spark):
    c = spark.range(1, N_CUSTOMERS + 1).withColumnRenamed("id", "customer_id")
    return c.select(
        "customer_id",
        F.when(F.col("customer_id") == 1, "Big Reseller Ltd")
         .otherwise(F.concat(F.lit("Customer "), F.col("customer_id").cast("string"))).alias("name"),
        _pick(_u(F.col("customer_id"), 1), COUNTRIES).alias("country"),
        F.date_add(F.lit("2022-01-01").cast("date"), F.floor(_u(F.col("customer_id"), 2) * 730).cast("int")).alias("signup_date"),
        F.when((F.col("customer_id") == 1) | (_u(F.col("customer_id"), 3) < 0.2), "business").otherwise("retail").alias("segment"),
    )

def make_products(spark):
    p = spark.range(1, N_PRODUCTS + 1).withColumnRenamed("id", "product_id")
    return p.select(
        "product_id",
        F.concat(F.lit("Product "), F.col("product_id").cast("string")).alias("name"),
        F.element_at(F.array(*[F.lit(c) for c in CATEGORIES]), (F.col("product_id") % 8 + 1).cast("int")).alias("category"),
        F.round((F.col("product_id") * 37) % 195 + 5.99, 2).alias("price"),
    )

def make_orders(spark, n=N_ORDERS, dirty=False):
    i = F.col("id")
    customer = (F.when(_u(i, 11) < 0.02, F.lit(None).cast("long"))                    # guest checkout: no customer
                 .when(_u(i, 10) < 0.30, F.lit(1).cast("long"))                       # the big reseller: 30% of orders
                 .otherwise((F.floor(_u(i, 12) * (N_CUSTOMERS - 1)) + 2).cast("long")))
    quantity = F.when(customer == 1, F.lit(5)).otherwise(F.floor(_u(i, 13) * 5) + 1).cast("int")   # the reseller always orders in bulk
    product = (F.floor(_u(i, 14) * N_PRODUCTS) + 1).cast("int")
    ts = F.lit("2024-01-01 00:00:00").cast("timestamp") + F.make_interval(secs=F.floor(_u(i, 15) * 91 * 86400).cast("int"))
    u = _u(i, 17)
    status = F.when(u < 0.55, "paid").when(u < 0.80, "shipped").when(u < 0.92, "cancelled").otherwise("returned")
    v = _u(i, 18)
    payment = F.when(v < 0.60, "card").when(v < 0.85, "cash").otherwise("wallet")
    df = spark.range(0, n, 1, 4).select(
        (i + 1).alias("order_id"), customer.alias("customer_id"), product.alias("product_id"), quantity.alias("quantity"),
        F.round(quantity * (((product * 37) % 195) + 5.99), 2).alias("amount"),
        status.alias("status"), payment.alias("payment_method"), ts.alias("order_ts"), F.md5(i.cast("string")).alias("tracking_code"))
    df = df.withColumn("order_date", F.to_date("order_ts"))
    if dirty:    # the monthly files of the final project contain mistakes on purpose
        df = df.withColumn("amount", F.when(_u(F.col("order_id"), 19) < 0.008, -F.col("amount")).otherwise(F.col("amount")))
        df = df.withColumn("quantity", F.when(_u(F.col("order_id"), 20) < 0.006, 0).otherwise(F.col("quantity")))
        df = df.unionByName(df.filter(_u(F.col("order_id"), 21) < 0.005))              # exact duplicates
    return df.select("order_id", "customer_id", "product_id", "quantity", "amount", "status",
                     "payment_method", "order_ts", "order_date", "tracking_code")

def _create_data(spark):
    os.makedirs("data/monthly", exist_ok=True)
    make_customers(spark).toPandas().to_csv("data/customers.csv", index=False)
    make_products(spark).toPandas().to_csv("data/products.csv", index=False)
    make_orders(spark).repartition(4).write.mode("overwrite").parquet("data/orders.parquet")
    dirty = make_orders(spark, dirty=True)
    wrong = _u(F.col("order_id"), 22) < 0.003          # a few orders land in the wrong month's file
    month = F.month("order_date")
    file_month = F.when(wrong, month % 3 + 1).otherwise(month)
    for m in (1, 2, 3):
        (dirty.filter(file_month == m).coalesce(1).write.mode("overwrite")
              .parquet(f"data/monthly/orders_2024-{m:02d}.parquet"))

# ---------------------------------------------------------------- the Spark session ----
def new_session(master="local[4]", driver_memory="2g", delta=False, avro=False, aqe=False, conf=None):
    builder = (SparkSession.builder.master(master).appName("day01_lab")
               .config("spark.driver.memory", driver_memory)
               .config("spark.ui.showConsoleProgress", "false")
               .config("spark.sql.adaptive.enabled", str(aqe).lower())   # off = exact job and stage counts
               .config("spark.sql.session.timeZone", "UTC")                # same dates and hours for everyone
               .config("spark.sql.warehouse.dir", os.path.abspath("spark-warehouse"))
               # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this.
               .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false"))
    for k, v in (conf or {}).items():
        builder = builder.config(k, v)
    if delta:
        from delta import configure_spark_with_delta_pip
        builder = (builder.config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
                          .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
        builder = configure_spark_with_delta_pip(builder, extra_packages=["org.apache.spark:spark-avro_2.13:4.2.0"] if avro else None)
    elif avro:
        builder = builder.config("spark.jars.packages", "org.apache.spark:spark-avro_2.13:4.2.0")
    s = builder.getOrCreate()
    s.sparkContext.setLogLevel("ERROR")
    return s

for _d in ("spark-warehouse", "metastore_db"):
    shutil.rmtree(_d, ignore_errors=True)               # a clean catalog on every start
spark = new_session()
if not all(os.path.exists(f) for f in ("data/orders.parquet", "data/customers.csv", "data/products.csv", "data/monthly/orders_2024-03.parquet")):
    print("creating the shop data (about 30 seconds, once)...")
    _create_data(spark)
print(f"Spark {spark.version} | UI: {spark.sparkContext.uiWebUrl}")

# ---------------------------------------------------------------- helpers ----------------
def _tracker():
    return spark.sparkContext.statusTracker()

def _run_group(action):
    sc = spark.sparkContext
    group = f"g{time.time_ns()}"
    sc.setJobGroup(group, group)
    try:
        action()
    finally:
        sc.setLocalProperty("spark.jobGroup.id", None)
    return [_tracker().getJobInfo(j) for j in sorted(_tracker().getJobIdsForGroup(group))]

def jobs_run(action):
    """How many Spark jobs calling action() starts."""
    return len(_run_group(action))

def stages_run(action):
    """How many stages those jobs had in total."""
    return sum(len(j.stageIds) for j in _run_group(action))

def last_stage_tasks(action):
    """Number of tasks in the last stage action() ran."""
    jobs = _run_group(action)
    return _tracker().getStageInfo(max(s for j in jobs for s in j.stageIds)).numTasks

def _api(path):
    sc = spark.sparkContext
    return json.load(urllib.request.urlopen(f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}{path}"))

api = _api   # read anything from the Spark UI's REST API, e.g. api("/executors")

def _stage_rows(action):
    jobs = _run_group(action)
    ids = {s for j in jobs for s in j.stageIds}
    return [s for s in _api("/stages?length=2000") if s["stageId"] in ids]

def shuffle_bytes(action):
    """Total shuffle bytes written by the stages action() ran."""
    return sum(s["shuffleWriteBytes"] for s in _stage_rows(action))

def spill_bytes(action):
    """(memory, disk) bytes spilled by the stages action() ran."""
    rows = _stage_rows(action)
    return sum(s["memoryBytesSpilled"] for s in rows), sum(s["diskBytesSpilled"] for s in rows)

def skew_ratio(action):
    """Slowest task / median task, in the stage with the most tasks that action() ran."""
    rows = _stage_rows(action)
    st = max(rows, key=lambda s: s["numTasks"])
    tasks = _api(f"/stages/{st['stageId']}/{st['attemptId']}/taskList?length=5000")
    times = sorted(t["taskMetrics"]["executorRunTime"] for t in tasks if t.get("taskMetrics"))
    return round(times[-1] / max(times[len(times) // 2], 1), 1)

def failed_attempts():
    """Failed task attempts in the most recent failed stage (from the Spark UI's REST API)."""
    failed = _api("/stages?status=failed")
    if not failed:
        return 0
    st = max(failed, key=lambda s: s["stageId"])
    tasks = _api(f"/stages/{st['stageId']}/{st['attemptId']}/taskList?length=500")
    return sum(1 for t in tasks if t["status"] == "FAILED")

def sql_metric(action, metric):
    """Value of one scan metric (for example 'number of files read') of the query action() runs."""
    desc = f"q{time.time_ns()}"
    spark.sparkContext.setJobDescription(desc)
    try:
        action()
    finally:
        spark.sparkContext.setJobDescription(None)
    for ex in _api("/sql?details=true&planDescription=false&length=2000"):
        if ex.get("description") == desc:
            vals = [m["value"] for n in ex["nodes"] for m in n["metrics"] if m["name"] == metric]
            return vals[0] if vals else None

def noop(df):
    """Run the whole query but throw the result away."""
    df.write.format("noop").mode("overwrite").save()

def plan(df, mode="formatted"):
    """explain() output as a string."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain(mode)
    return buf.getvalue()

def count_nodes(df, *names):
    """How often each operator name appears in the simple plan."""
    text = plan(df, "simple")
    return {n: len(re.findall(r"\b" + re.escape(n) + r"\b", text)) for n in names}

def optimized(df):
    """The optimized logical plan as a string."""
    return df._jdf.queryExecution().optimizedPlan().toString()

def operators(df):
    """Operator names of the physical plan tree, top to bottom."""
    tree = plan(df).split("\n\n")[0].splitlines()[1:]
    return [re.sub(r"\s*\(\d+\)$", "", line).strip(" +-*:") for line in tree]

def rules_fired(df):
    """Names of the Catalyst rules that changed this query's plan."""
    qe = df._jdf.queryExecution(); qe.executedPlan()
    it, fired = qe.tracker().rules().iterator(), set()
    while it.hasNext():
        kv = it.next()
        if kv._2().numEffectiveInvocations() > 0:
            fired.add(kv._1().split(".")[-1].split("$")[-1])
    return fired

def estimate(df):
    """(sizeInBytes, rowCount or None) the optimizer estimates for df."""
    stats = df._jdf.queryExecution().optimizedPlan().stats()
    return int(str(stats.sizeInBytes())), (int(str(stats.rowCount().get())) if stats.rowCount().isDefined() else None)

def partition_sizes(df):
    """Rows per partition, largest first."""
    return sorted((r["count"] for r in df.groupBy(F.spark_partition_id().alias("p")).count().collect()), reverse=True)

def timeit(action):
    start = time.perf_counter(); action(); return round(time.perf_counter() - start, 2)

def folder_mb(path):
    files = [os.path.join(d, f) for d, _, fs in os.walk(path) for f in fs] if os.path.isdir(path) else [path]
    return round(sum(os.path.getsize(f) for f in files) / 1e6, 1)

# ---------------------------------------------------------------- checks -----------------
_EXPECTED = {
 "1.1": "8c52e654fc230bf1",
 "1.2": "6cce36d9f8a9e151",
 "1.3": "6c94e35ccc352d4e",
 "1.4": "5feceb66ffc86f38",
 "1.5": "fbe86ebdfe081051",
 "1.6": "30f472ada3ac0bc3",
 "1.7": "6767a9a3437def51",
 "1.8": "56855b425714a7bc",
 "1.9": "4b227777d4dd1fc6",
 "1.10": "f3e10d9b29bd8d31",
 "1.11": "f9818af02f6b42f3",
 "1.12": "c38dca8489bc92fc",
 "1.13": "20bcd19ab13c79e6",
 "1.14": "ccf98af88419749c",
 "1.15": "4e3970018fa50092"
}
_RECORD = os.environ.get("LAB_RECORD")

def _norm(v):
    if isinstance(v, bool) or v is None: return v
    if isinstance(v, int): return v
    if isinstance(v, float): return round(v, 2)
    if isinstance(v, str): return v.strip()
    if isinstance(v, dict): return {str(k): _norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
    if isinstance(v, (list, tuple)): return [_norm(x) for x in v]
    if isinstance(v, (set, frozenset)): return sorted((_norm(x) for x in v), key=json.dumps)
    if hasattr(v, "item"): return _norm(v.item())
    if type(v).__name__ == "Decimal": return round(float(v), 2)
    raise TypeError(f"check() needs plain Python values, got {type(v).__name__}")

def check(task, value):
    normal = _norm(value)
    if _RECORD:
        rec = json.load(open(_RECORD)) if os.path.exists(_RECORD) else {}
        rec[task] = normal; json.dump(rec, open(_RECORD, "w"))
    ok = hashlib.sha256(json.dumps(normal).encode()).hexdigest()[:16] == _EXPECTED.get(task)
    print(f"✅ {task} correct" if ok else f"❌ {task} not yet: check your code and the question again")

def check_true(task, condition, hint=""):
    print(f"✅ {task} correct" if condition else f"❌ {task} not yet. {hint}")


## A. The session and the data

**1.1 Which master URL and how many default parallelism does your Spark session use?** Put both values in *info*.

In [ ]:
info = (spark.sparkContext.master, spark.sparkContext.defaultParallelism)
info

In [ ]:
check("1.1", info)

**1.2 Load `data/orders.parquet` into *orders*. How many orders are there?** Put the number in *n_orders*.

In [ ]:
orders = spark.read.parquet("data/orders.parquet")
n_orders = orders.count()
n_orders

In [ ]:
check("1.2", n_orders)

**1.3 Load `data/customers.csv` into *customers* and `data/products.csv` into *products*, with a header and an explicit schema (no `inferSchema`):**
* customers: `customer_id long, name string, country string, signup_date date, segment string`
* products: `product_id long, name string, category string, price double`

How many customers are in the *business* segment? Put it in *n_business*.

In [ ]:
customers = (spark.read.option("header", True)
             .schema("customer_id long, name string, country string, signup_date date, segment string")
             .csv("data/customers.csv"))
products = (spark.read.option("header", True)
            .schema("product_id long, name string, category string, price double")
            .csv("data/products.csv"))
n_business = customers.filter(F.col("segment") == "business").count()
n_business

In [ ]:
check("1.3", n_business)

## B. Lazy evaluation: transformations vs. actions

**1.4 Create *big_orders*: orders with *amount* over 500, with the columns *order_id*, *customer_id*, *amount*, *order_date* and a new column *tax* (12% of *amount*, rounded to 2 decimals). How many Spark jobs does creating it start?**

Guess first, then measure with `jobs_run`.

In [ ]:
def build():
    return (orders.filter(F.col("amount") > 500)
                  .select("order_id", "customer_id", "amount", "order_date")
                  .withColumn("tax", F.round(F.col("amount") * 0.12, 2)))

jobs_building = jobs_run(build)
big_orders = build()
jobs_building

In [ ]:
check("1.4", jobs_building)

**1.5 Which of these start a Spark job on *big_orders*: `printSchema`, `explain`, `columns`, `count`, `show`, writing to the `noop` sink?**

Guess first, then measure each with `jobs_run`. Put the numbers in a dict *job_counts*, one key per call.

In [ ]:
job_counts = {
    "printSchema": jobs_run(lambda: big_orders.printSchema()),
    "explain": jobs_run(lambda: big_orders.explain()),
    "columns": jobs_run(lambda: big_orders.columns),
    "count": jobs_run(lambda: big_orders.count()),
    "show": jobs_run(lambda: big_orders.show(5)),
    "write_noop": jobs_run(lambda: noop(big_orders)),
}
job_counts

In [ ]:
check("1.5", job_counts)

**1.6 What is the average *amount* of *big_orders*?** Round to 2 decimals and put it in *avg_big*.

In [ ]:
avg_big = round(big_orders.agg(F.avg("amount")).first()[0], 2)
avg_big

In [ ]:
check("1.6", avg_big)

## C. Narrow vs. wide: stages and tasks

**1.7 How many stages does each query run?**
* a: orders with *amount* over 500, only *amount* and *quantity*, written with `noop`
* b: total *amount* per *status*, collected
* c: all orders sorted by *amount*, written with `noop`

Guess first, then measure with `stages_run`. Put the numbers in a dict *stages* with the keys `a`, `b`, `c`. Also measure how many *jobs* query c starts (*jobs_c*).

In [ ]:
stages = {
    "a": stages_run(lambda: noop(orders.filter("amount > 500").select("amount", "quantity"))),
    "b": stages_run(lambda: orders.groupBy("status").agg(F.sum("amount")).collect()),
    "c": stages_run(lambda: noop(orders.orderBy("amount"))),
}
jobs_c = jobs_run(lambda: noop(orders.orderBy("amount")))
stages, jobs_c

In [ ]:
check("1.7", [stages, jobs_c])

**1.8 How many tasks does the last stage of query b have with the default `spark.sql.shuffle.partitions`, and with it set to 8?** Use `last_stage_tasks`, put both numbers in *tasks* as `(default, eight)`, then set it back to 200.

In [ ]:
b = lambda: orders.groupBy("status").agg(F.sum("amount")).collect()
tasks_default = last_stage_tasks(b)
spark.conf.set("spark.sql.shuffle.partitions", 8)
tasks_8 = last_stage_tasks(b)
spark.conf.set("spark.sql.shuffle.partitions", 200)
tasks = (tasks_default, tasks_8)
tasks

In [ ]:
check("1.8", tasks)

**1.9 How many partitions does *orders* have?** Put it in *read_tasks*, then find the same number of tasks in the first stage of query b in the Spark UI.

In [ ]:
read_tasks = orders.rdd.getNumPartitions()
read_tasks

In [ ]:
check("1.9", read_tasks)

## D. A real question, and the same as an app

**1.10 Which 5 countries have the most paid orders?**
* use only orders with *status* `paid`
* join *orders.customer_id* to *customers.customer_id* (guest orders drop out)
* *top_countries* = list of `(country, count)`, largest first, ties by country name

Then measure the jobs and stages of collecting it into *jobs_stages* as `(jobs, stages)`.

In [ ]:
top_q = (orders.filter(F.col("status") == "paid")
         .join(customers, "customer_id")
         .groupBy("country").count()
         .orderBy(F.desc("count"), "country").limit(5))
top_countries = [(r["country"], r["count"]) for r in top_q.collect()]
jobs_stages = (jobs_run(top_q.collect), stages_run(top_q.collect))
top_countries, jobs_stages

In [ ]:
check("1.10", [top_countries, jobs_stages])

**1.11 Write the app *top_products.py*: find the 3 products with the highest total *quantity* in paid orders (columns *product_id* and *total_qty*, ties by *product_id*) and save them as CSV with a header to `out/top_products`. Run it with `spark-submit`.**

In [ ]:
%%writefile top_products.py
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("top-products").getOrCreate()
orders = spark.read.parquet("data/orders.parquet")
top = (orders.filter(F.col("status") == "paid")
       .groupBy("product_id").agg(F.sum("quantity").alias("total_qty"))
       .orderBy(F.desc("total_qty"), "product_id").limit(3))
top.write.mode("overwrite").option("header", True).csv("out/top_products")
spark.stop()

In [ ]:
SPARK_SUBMIT = os.path.join(os.path.dirname(sys.executable), "spark-submit")
!{SPARK_SUBMIT} --master "local[2]" top_products.py 2>/dev/null
top_products = [(int(r["product_id"]), int(r["total_qty"])) for r in spark.read.option("header", True).csv("out/top_products").orderBy(F.desc(F.col("total_qty").cast("int")), "product_id").collect()]
check("1.11", top_products)

## E. Coding practice: DataFrame basics

Three small problems with columns, conditions, dates and missing values. Write them from scratch.

**1.12 Give every order a size label in a new column *size*: `small` if *amount* is under 50, `medium` if under 300, otherwise `large`. How many orders are there of each size?** Put the counts in a dict *size_counts* (`{size: count}`).

In [ ]:
sized = orders.withColumn("size", F.when(F.col("amount") < 50, "small").when(F.col("amount") < 300, "medium").otherwise("large"))
size_counts = {r["size"]: r["count"] for r in sized.groupBy("size").count().collect()}
size_counts

In [ ]:
check("1.12", size_counts)

**1.13 Look at the customer names. Customers are named `Customer <number>`; one is not (Big Reseller Ltd).**
* add a column *number* with the number from the name (`null` for the reseller; careful: in Spark 4 casting a non-number such as an empty string to `int` raises an error)
* count the customers who signed up on a weekend (Saturday or Sunday, use `dayofweek`)

Put `[weekend_signups, no_number]` in *result*, where *no_number* is the number of customers whose *number* is null.

In [ ]:
named = customers.withColumn(
    "number", F.when(F.col("name").rlike("^Customer [0-9]+$"), F.regexp_extract("name", r"(\d+)$", 1).cast("int")))
weekend_signups = customers.filter(F.dayofweek("signup_date").isin(1, 7)).count()
no_number = named.filter(F.col("number").isNull()).count()
result = [weekend_signups, no_number]
result

In [ ]:
check("1.13", result)

**1.14 Guest checkout orders have no *customer_id*.**
* how many orders are guest orders, and what is their total *amount* (2 decimals)?
* make *with_guest*: *orders* with the missing *customer_id* replaced by `0`. How many different *customer_id* values does it have?

Put `[guest_orders, guest_amount, distinct_ids]` in *result*.

In [ ]:
guest = orders.filter(F.col("customer_id").isNull())
with_guest = orders.na.fill({"customer_id": 0})
result = [guest.count(), round(guest.agg(F.sum("amount")).first()[0], 2), with_guest.select("customer_id").distinct().count()]
result

In [ ]:
check("1.14", result)

## F. Fault tolerance

**1.15 How many times does Spark try a task that keeps failing: with master `local[4]`, and with `local[4,3]`?**
* *risky* raises an error for order 777777 only; run `orders.agg(F.sum(risky("order_id"))).collect()` and catch the error
* read the failed attempts with `failed_attempts()`
* restart with `spark.stop()` and `spark = new_session("local[4,3]")`, load *orders* again, run it again

Put `(attempts_local, attempts_retry)` in *attempts*.

(Why not `select(risky(...)).count()`? `count()` needs no columns, so Spark never calls the UDF.)

In [ ]:
from pyspark.sql.functions import udf
risky = udf(lambda i: 1 / 0 if i == 777777 else i, "long")

try:
    orders.agg(F.sum(risky("order_id"))).collect()
except Exception:
    pass
attempts_local = failed_attempts()

spark.stop()
spark = new_session("local[4,3]")
orders = spark.read.parquet("data/orders.parquet")
try:
    orders.agg(F.sum(risky("order_id"))).collect()
except Exception:
    pass
attempts = (attempts_local, failed_attempts())
attempts

In [ ]:
check("1.15", attempts)

---
Done? In the Theory part of Day 1, answer the red checklist topics aloud, pointing at what you measured here.